# 02 — NLP Preprocessing

Input: `unified_recipes.parquet` from `01_data_cleaning.ipynb`.

Goal: build one clean text field per recipe (title + ingredients + description + instructions) ready for SBERT embedding.

**Known risk going in:** all-MiniLM-L6-v2 truncates at 256 tokens. Full instructions text will exceed this for many recipes. This notebook measures how bad that is before the next step, rather than assuming it's fine.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import re
import numpy as np

pd.set_option('display.max_colwidth', 100)

In [ ]:
PROCESSED_DIR = Path.cwd().parent / "datasets" / "processed"

df = pd.read_parquet(PROCESSED_DIR / "unified_recipes.parquet")
print(df.shape)
df.head(3)

## 1. Text cleaning helpers

In [ ]:
def clean_text(text: str) -> str:
    """Basic normalization: lowercase, strip extra whitespace/punctuation noise."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)          # strip URLs (RecipeNLG has scrape artifacts)
    text = re.sub(r"[^a-z0-9\s,.\-']", " ", text)            # keep basic punctuation only
    text = re.sub(r"\s+", " ", text).strip()
    return text


def join_list_field(items) -> str:
    """Join a list-typed column (ingredients/instructions) into one string."""
    if isinstance(items, (list, np.ndarray)):
        return ". ".join(str(i) for i in items)
    return ""


sample = df.iloc[0]
print(clean_text(sample["title"]))
print(join_list_field(sample["ingredients"])[:200])

## 2. Build combined text field

Full text: title + ingredients + description + instructions, as decided.

In [ ]:
def build_combined_text(row) -> str:
    title = clean_text(row["title"])
    ingredients = clean_text(join_list_field(row["ingredients"]))
    description = clean_text(row["description"])
    instructions = clean_text(join_list_field(row["instructions"]))

    parts = [p for p in [title, ingredients, description, instructions] if p]
    return " . ".join(parts)


df["combined_text"] = df.apply(build_combined_text, axis=1)

# Drop rows that ended up with essentially no usable text
before = len(df)
df = df[df["combined_text"].str.len() > 10]
print(f"{before} -> {len(df)} rows after dropping empty combined_text")

df[["title", "combined_text"]].head(2)

## 3. Token length check (the important part)

MiniLM's tokenizer isn't loaded yet at this stage of the pipeline (that's the embedding notebook's job), so this uses a rough word-count proxy — actual subword token count runs ~1.3x word count for English text. Good enough to see the shape of the problem now; the embedding notebook will report exact token counts when it tokenizes for real.

In [ ]:
df["word_count"] = df["combined_text"].str.split().map(len)
df["approx_token_count"] = (df["word_count"] * 1.3).astype(int)

print(df["approx_token_count"].describe())

MINILM_LIMIT = 256
pct_over_limit = (df["approx_token_count"] > MINILM_LIMIT).mean() * 100
print(f"\n~{pct_over_limit:.1f}% of recipes exceed MiniLM's 256-token limit "
      f"and will be silently truncated at embedding time.")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.hist(df["approx_token_count"], bins=60, range=(0, 800))
plt.axvline(MINILM_LIMIT, color="red", linestyle="--", label="MiniLM 256-token limit")
plt.xlabel("approx token count")
plt.ylabel("# recipes")
plt.title("Combined text length vs MiniLM truncation point")
plt.legend()
plt.show()

### Decision point

Look at the `pct_over_limit` number above before moving to the embedding notebook:

- **If it's small (~<15%)** — MiniLM is fine as-is, truncation only clips a minority of long-tail recipes at the instructions end. Proceed with MiniLM.
- **If it's large (~>40%)** — most recipes are losing their back-half instructions at embedding time, which weakens semantic matching for anything distinguished by later steps (technique-heavy recipes especially). Two real options: switch to `all-mpnet-base-v2` (384 tokens, slower, more accurate) or drop instructions from the embedded text and keep them only for display — title+ingredients+description is usually under the limit and instructions rarely differentiate similarity much anyway.

Don't skip this check — this is the same category of mistake as DataQ's unvalidated 0.7/0.3 weighting: picking a config without checking it against the actual data.

## 4. Save preprocessed dataset

In [ ]:
output_cols = [c for c in df.columns if c not in ["word_count", "approx_token_count"]]
df_out = df[output_cols].reset_index(drop=True)

df_out.to_parquet(PROCESSED_DIR / "nlp_preprocessed_recipes.parquet", index=False)
print(f"Saved {len(df_out)} rows to {PROCESSED_DIR / 'nlp_preprocessed_recipes.parquet'}")

## Next notebook: `03_feature_engineering.ipynb`

This is where `combined_text` gets fed into SBERT for embeddings, nutrition columns get scaled/normalized, and the FAISS index gets built.